### SPARQL Query Analysis

This notebook fetches SPARQL queries from the LSQ endpoint, analyzes duplicates, queries the DBpedia endpoint, and saves successful queries for further analysis.

In [ ]:
import os
import time
import json
import pandas as pd
import requests
from tqdm import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [ ]:

# ---------- CONFIG ----------
LSQ_ENDPOINT = "http://lsq.aksw.org/sparql"
DBPEDIA_ENDPOINT = "https://dbpedia.data.dice-research.org/sparql"  # target DBpedia endpoint
BATCH_SIZE = 1000
NUM_BATCHES = 50           # reduce for quick sampled runs
SAMPLE_SIZE = None         # set to int to cap total queries after fetch (None = no cap)
OUTPUT_FILE = "queries_with_answers_sampled.json"
MIN_RESULTS_PER_QUERY = 1
MAX_BINDINGS_PER_QUERY = 50
REQUEST_TIMEOUT = 15
DELAY_BETWEEN_REQUESTS = 0.1
SAVE_EVERY = 50


In [ ]:
# ---------- Fetch SELECT queries from LSQ (filtered for DBpedia) ----------
QUERY_TEMPLATE = """
PREFIX lsqv: <http://lsq.aksw.org/vocab#>
PREFIX sd: <http://www.w3.org/ns/sparql-service-description#>

SELECT DISTINCT ?text
WHERE {{
  ?query lsqv:hasRemoteExec/lsqv:endpoint <https://dbpedia.org/sparql> .
  ?query lsqv:text ?text .
  FILTER regex(str(?text), "^SELECT", "i")
}}
LIMIT {limit}
OFFSET {offset}
"""

def fetch_select_queries_lsqs(num_batches=NUM_BATCHES, batch_size=BATCH_SIZE):
    sparql = SPARQLWrapper(LSQ_ENDPOINT)
    all_queries = []
    for i in range(num_batches):
        offset = i * batch_size
        sparql.setQuery(QUERY_TEMPLATE.format(limit=batch_size, offset=offset))
        sparql.setReturnFormat(JSON)
        try:
            results = sparql.query().convert()
            texts = [b["text"]["value"] for b in results.get("results", {}).get("bindings", [])]
            all_queries.extend(texts)
            print(f"Fetched {len(texts)} queries (batch {i+1}/{num_batches})")
        except Exception as e:
            print(f"LSQ fetch error at offset {offset}: {e}")
        time.sleep(0.2)
    if SAMPLE_SIZE:
        all_queries = all_queries[:SAMPLE_SIZE]
    return pd.DataFrame({"query": all_queries})


In [ ]:
# Uncomment to run fetch
df = fetch_select_queries_lsqs()
df.to_csv("sampled_select_queries.csv", index=False)

In [ ]:
# ---------- Load sampled queries ----------
if not os.path.exists("sampled_select_queries.csv"):
    raise FileNotFoundError("Run the LSQ fetch step first and save sampled_select_queries.csv")

df = pd.read_csv("sampled_select_queries.csv")
print(f"Loaded {len(df)} sampled queries")
df_unique = df.drop_duplicates(subset="query").reset_index(drop=True)
print(f"Unique queries after dedup: {len(df_unique)}")


In [ ]:
# ---------- Prepare HTTP session with retries ----------
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=(429,500,502,503,504))
session.mount("https://", HTTPAdapter(max_retries=retries))
HEADERS = {"Accept": "application/sparql-results+json"}

def query_dbpedia(q):
    try:
        r = session.get(DBPEDIA_ENDPOINT, params={"query": q}, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"_error": str(e)}

In [ ]:
# ---------- Run queries and collect successful ones (sample-friendly) ----------
successful = []
for i, row in tqdm(df_unique.iterrows(), total=len(df_unique), desc="Querying DBpedia"):
    sparql = str(row["query"]).strip()
    if not sparql:
        continue
    res = query_dbpedia(sparql)
    if "_error" in res:
        # skip / optionally log
        continue
    bindings = res.get("results", {}).get("bindings", []) or []
    if len(bindings) >= MIN_RESULTS_PER_QUERY:
        successful.append({
            "sparql_query": sparql,
            "results_count": len(bindings),
            "results": bindings[:MAX_BINDINGS_PER_QUERY]
        })
    time.sleep(DELAY_BETWEEN_REQUESTS)
    if SAVE_EVERY and len(successful) and len(successful) % SAVE_EVERY == 0:
        with open(OUTPUT_FILE + ".partial", "w", encoding="utf-8") as f:
            json.dump(successful, f, ensure_ascii=False, indent=2)


In [ ]:
# ---------- Final save ----------
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(successful, f, ensure_ascii=False, indent=2)

print(f"Saved {len(successful)} successful queries to {OUTPUT_FILE}")